<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/day-17-metadata-filtering/rag-metadata-filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q faiss-cpu google-generativeai

import time
import numpy as np
import faiss
import google.generativeai as genai
from google.colab import userdata
from datetime import datetime

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
chat_model = genai.GenerativeModel('gemini-3.6-flash')
print("Setup done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 25.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Setup done.


In [2]:
knowledge_base = [
    {
        "text": "Nimbus Robotics was founded in 2031 by engineer Priya Kalathil in Pune, India.",
        "source": "company_history.pdf", "category": "company_background", "date": "2031-03-15", "document_type": "reference"
    },
    {
        "text": "Nimbus Robotics' flagship product is the Aster-7, a warehouse picking robot with a 99.2% accuracy rate.",
        "source": "product_sheet_aster7.pdf", "category": "product", "date": "2033-01-10", "document_type": "technical"
    },
    {
        "text": "The Aster-7 uses a proprietary gripper called FlexGrip, adjusting pressure using 12 micro-sensors per finger.",
        "source": "product_sheet_aster7.pdf", "category": "product", "date": "2033-01-10", "document_type": "technical"
    },
    {
        "text": "Nimbus Robotics reported revenue of 340 million rupees in fiscal year 2033.",
        "source": "annual_report_2033.pdf", "category": "finance", "date": "2033-12-31", "document_type": "reference"
    },
    {
        "text": "Nimbus Robotics' main competitor is Solace Automation, founded a year earlier in 2030.",
        "source": "market_overview.pdf", "category": "market", "date": "2032-06-01", "document_type": "marketing"
    },
    {
        "text": "The Aster-7's battery lasts 14 hours on a single charge and recharges fully in 40 minutes.",
        "source": "product_sheet_aster7.pdf", "category": "product", "date": "2033-01-10", "document_type": "technical"
    },
    {
        "text": "Nimbus Robotics employs 212 people across three offices: Pune, Bengaluru, and Singapore.",
        "source": "company_history.pdf", "category": "company_background", "date": "2033-06-01", "document_type": "reference"
    },
    {
        "text": "The company's CTO, Rohan Mehta, previously led robotics research at a university lab for eight years.",
        "source": "leadership_bios.pdf", "category": "company_background", "date": "2031-04-01", "document_type": "reference"
    },
    {
        "text": "Nimbus Robotics' next product, the Aster-8, is scheduled for release in early 2035.",
        "source": "product_roadmap_2034.pdf", "category": "product", "date": "2034-09-01", "document_type": "marketing"
    },
    {
        "text": "The Aster-7 has been deployed in over 60 warehouses across South and Southeast Asia.",
        "source": "market_overview.pdf", "category": "market", "date": "2033-08-15", "document_type": "marketing"
    }
]

print(f"Knowledge base size: {len(knowledge_base)} documents")

Knowledge base size: 10 documents


In [3]:
def get_embeddings(texts, model="models/gemini-embedding-001"):
    embeddings = []
    for text in texts:
        result = genai.embed_content(model=model, content=text)
        embeddings.append(result['embedding'])
        time.sleep(1)
    return embeddings

texts_only = [doc["text"] for doc in knowledge_base]
corpus_embeddings = get_embeddings(texts_only)
corpus_vectors = np.array(corpus_embeddings, dtype=np.float32)

dimension = corpus_vectors.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(corpus_vectors)

print(f"FAISS index size: {index.ntotal} vectors")
print(f"Metadata store size: {len(knowledge_base)} entries (index position i <-> knowledge_base[i])")

FAISS index size: 10 vectors
Metadata store size: 10 entries (index position i <-> knowledge_base[i])


In [4]:
def filtered_retrieve(query, filters=None, top_k=3, search_pool=10):
    """
    Retrieves top_k documents matching the query, optionally constrained by metadata filters.
    filters: dict like {"category": "product"} or {"date_after": "2033-01-01"}

    Strategy: over-fetch a larger pool from FAISS (search_pool), then apply metadata
    filters as a post-filter, then trim to top_k. This is the standard "retrieve-then-filter"
    pattern when the vector store itself doesn't support filtering natively.
    """
    query_embedding = get_embeddings([query])[0]
    query_vector = np.array([query_embedding], dtype=np.float32)

    distances, indices = index.search(query_vector, search_pool)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        doc = knowledge_base[idx]

        # --- category filter ---
        if filters and "category" in filters:
            if doc["category"] != filters["category"]:
                continue

        # --- date filter (exclude documents older than cutoff) ---
        if filters and "date_after" in filters:
            doc_date = datetime.strptime(doc["date"], "%Y-%m-%d")
            cutoff = datetime.strptime(filters["date_after"], "%Y-%m-%d")
            if doc_date < cutoff:
                continue

        # --- document_type filter ---
        if filters and "document_type" in filters:
            if doc["document_type"] != filters["document_type"]:
                continue

        results.append({"text": doc["text"], "distance": float(dist), "metadata": doc})
        if len(results) >= top_k:
            break

    return results

In [5]:
test_queries = [
    {"query": "What products does Nimbus Robotics make?", "filters": {"category": "product"}},
    {"query": "What is the company's financial performance?", "filters": {"category": "finance"}},
    {"query": "Tell me about the technical specs of the Aster-7", "filters": {"document_type": "technical"}},
    {"query": "What are the latest developments at Nimbus Robotics?", "filters": {"date_after": "2033-01-01"}},
    {"query": "Who leads the company?", "filters": {"category": "company_background"}}
]

for t in test_queries:
    print("=" * 70)
    print(f"QUERY: {t['query']}")
    print(f"FILTER: {t['filters']}\n")

    print("WITHOUT filter:")
    unfiltered = filtered_retrieve(t['query'], filters=None, top_k=3)
    for r in unfiltered:
        print(f"  [dist={r['distance']:.4f}, category={r['metadata']['category']}] {r['text']}")

    print("\nWITH filter:")
    filtered = filtered_retrieve(t['query'], filters=t['filters'], top_k=3)
    for r in filtered:
        print(f"  [dist={r['distance']:.4f}, category={r['metadata']['category']}] {r['text']}")
    print()

QUERY: What products does Nimbus Robotics make?
FILTER: {'category': 'product'}

WITHOUT filter:
  [dist=0.3978, category=company_background] Nimbus Robotics employs 212 people across three offices: Pune, Bengaluru, and Singapore.
  [dist=0.4119, category=product] Nimbus Robotics' flagship product is the Aster-7, a warehouse picking robot with a 99.2% accuracy rate.
  [dist=0.4280, category=product] Nimbus Robotics' next product, the Aster-8, is scheduled for release in early 2035.

WITH filter:
  [dist=0.4119, category=product] Nimbus Robotics' flagship product is the Aster-7, a warehouse picking robot with a 99.2% accuracy rate.
  [dist=0.4280, category=product] Nimbus Robotics' next product, the Aster-8, is scheduled for release in early 2035.
  [dist=0.8113, category=product] The Aster-7 uses a proprietary gripper called FlexGrip, adjusting pressure using 12 micro-sensors per finger.

QUERY: What is the company's financial performance?
FILTER: {'category': 'finance'}

WITHOUT filte